In [79]:
# papermill parameters

device_map = 'cpu'
cross_learning = True
lookback = 60
lookforward = 1

input_ohlcv_file='binance-1d-spot-stable-pairs.parquet'
input_gjr_file='binance-1d-spot-volatility-forecast.parquet'

output_pred_file='chronos-2-predictions.parquet'
output_eval_file='chronos-2-eval.parquet'

# fundamental metrics
use_open = 1
use_close = 1
use_high = 1
use_low = 1
use_base_volume = 1
use_quote_volume = 1

# derived, past metrics
use_ofi = 0
use_momentum_window = 30
use_sigma_rs = 0

# derived, forecasted metrics
use_gjr_garch_forecast_horizon = 7

In [80]:
print(f'''
device_map = {device_map}
cross_learning = {cross_learning}
lookback = {lookback}
lookforward = {lookforward}

input_ohlcv_file = {input_ohlcv_file}
input_gjr_file = {input_gjr_file}

output_pred_file = {output_pred_file}
output_eval_file = {output_eval_file}

# fundamental metrics
use_open = {use_open}
use_close = {use_close}
use_high = {use_high}
use_low = {use_low}
use_base_volume = {use_base_volume}
use_quote_volume = {use_quote_volume}

# derived, past metrics
use_ofi = {use_ofi}
use_momentum_window = {use_momentum_window}
use_sigma_rs = {use_sigma_rs}

# derived, forecasted metrics
use_gjr_garch_forecast_horizon = {use_gjr_garch_forecast_horizon}
''')


device_map = cpu
cross_learning = True
lookback = 60
lookforward = 1

input_ohlcv_file = binance-1d-spot-stable-pairs.parquet
input_gjr_file = binance-1d-spot-volatility-forecast.parquet

output_pred_file = chronos-2-predictions.parquet
output_eval_file = chronos-2-eval.parquet

# fundamental metrics
use_open = 1
use_close = 1
use_high = 1
use_low = 1
use_base_volume = 1
use_quote_volume = 1

# derived, past metrics
use_ofi = 0
use_momentum_window = 30
use_sigma_rs = 0

# derived, forecasted metrics
use_gjr_garch_forecast_horizon = 7



In [81]:
# volatility, taker buy/sell ratio, momentum, auto correlation, level

import polars as pl
from arch import arch_model
import numpy as np
from tqdm import tqdm

ohlcv = pl.read_parquet(input_ohlcv_file).sort(['symbol', 'ts'])
gjr = pl.read_parquet(input_gjr_file).sort(['symbol', 'ts'])

df = (
    ohlcv.join(gjr, on=['symbol','ts'])
        .filter(
            (pl.col('open') > 0) & (pl.col('high') > 0) &
            (pl.col('low') > 0) & (pl.col('close') > 0) &
            (pl.col('volume') > 0))
        .with_columns([
            # todays simple return (target)
            pl.col('close')
                .diff()
                .over('symbol')
                .alias('ret')])
        .select([
            # base columns
            pl.col('ts'),
            pl.col('symbol'),
            pl.col('ret'),                
            
            # ohlcv data
            pl.when(use_open > 0).then(pl.col('open')).otherwise(np.nan).alias('open'),
            pl.when(use_high > 0).then(pl.col('high')).otherwise(np.nan).alias('high'),
            pl.when(use_low > 0).then(pl.col('low')).otherwise(np.nan).alias('low'),
            pl.when(use_close > 0).then(pl.col('close')).otherwise(np.nan).alias('close'),
            pl.when(use_base_volume > 0).then(pl.col('volume')).otherwise(np.nan).alias('base_volume'),
            #pl.when(use_quote_volume > 0).then(pl.col('qty_volume')).otherwise(np.nan).alias('quote_volume'),
        
            # rogers-satchell volatility
            pl.when(use_sigma_rs > 0)
                .then(
                    (
                        ((pl.col('high') / pl.col('close')).log() * (pl.col('high') / pl.col('open')).log()) +
                        ((pl.col('low') / pl.col('close')).log() * (pl.col('low') / pl.col('open')).log())
                    )    
                    .sqrt())
                .otherwise(np.nan)
                .alias('sigma_rs'),
        
            #(pl.col('qty_volume') / pl.col('volume')).alias('vwap'),
            pl.when(use_ofi > 0)
                .then((2 * pl.col('taker_buy_base_asset_volume') / pl.col('volume')) - 1)
                .otherwise(np.nan)
                .alias('ofi'),
        
            # n-day momentum
            pl.when(use_momentum_window > 0)
                .then(pl.col('ret').rolling_sum(window_size=use_momentum_window).over('symbol'))
                .otherwise(np.nan)
                .alias('momentum'),
        
            # gjr-garch
            pl.when(use_gjr_garch_forecast_horizon > 0)
                .then(pl.col('sigma_gjr'))
                .otherwise(np.nan)
                .alias('sigma_gjr'),
            ])
)

df = (
    df.select([
        pl.col(c) for c in df.columns 
        if not (df[c].is_nan().all() if df[c].dtype.is_float() else False)])
    .drop_nulls()
)

In [49]:
# inference with chronos-2

import numpy as np
import pandas as pd
from chronos import Chronos2Pipeline
import datetime as dt
from tqdm import tqdm

pipeline = Chronos2Pipeline.from_pretrained("amazon/chronos-2", device_map=device_map)
pred = []

for td in tqdm(df['ts'].unique().sort()[:100]):
    lb = td - dt.timedelta(days=lookback)

    win = (df
        .filter((pl.col('ts') >= lb) & (pl.col('ts') <= td))
        .drop_nulls()
        .upsample(time_column="ts", every="1d", group_by="symbol")
        .with_columns(pl.all().forward_fill())
        .filter(
            (pl.len().over("symbol") >= lookback) & (pl.col("ts").max().over("symbol") == td)  
        )
    )

    if win.is_empty():
        continue
    
    # Generate predictions with covariates
    pdf = pipeline.predict_df(
        win.to_pandas(),
        prediction_length=lookforward,
        quantile_levels=[0.5],
        id_column="symbol",
        timestamp_column="ts",
        target="ret",
        validate_inputs=True,
        cross_learning=cross_learning,
    )

    pdf = (
        pl.from_pandas(pdf)
        .group_by("symbol")
        .agg([
            pl.col("0.5").sum().alias("ret"),
            pl.col("ts").max().alias("hori")
        ])
        .with_columns(pl.lit(td).alias("ts"))
    )

    pred.append(pdf)

pl.concat(pred).write_parquet(output_pred_file)

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:03<00:00, 31.72it/s]


In [82]:
# compute performance

import matplotlib.pyplot as plt

df = (df
    .join(
        pl.read_parquet(output_pred_file)
            .select([pl.col('ts'),pl.col('symbol'),pl.col('ret').alias('pred')]),
        on=['ts','symbol'],
        how='inner')
    .with_columns([
        pl.col('pred')
            .qcut(
                10, 
                allow_duplicates=True, 
                labels=list(map(lambda n: f'{n}', range(1,11))))
            .over('ts')
            .alias('rank')])
    .filter([
        pl.col('rank') == '10', pl.col('pred') > 0])
    .with_columns([
        (pl.col('base_volume') / pl.col('base_volume').sum().over(['ts'])).alias('weight')])
    .sort(['ts','symbol'])
    .with_columns([
        pl.col('weight').alias('initial'),
        (pl.col('weight') * pl.col('ret').exp()).alias('eod')])
    .with_columns([
        ((pl.col('eod').abs() + pl.col('initial').abs()) * 0.002).alias('fee')])
    .with_columns([
        (pl.col('eod') - pl.col('initial') - pl.col('fee')).alias('pnl')]))

res = (df
    .sort('ts')
    .group_by('ts')
    .agg([
        (1 + pl.col('pnl').sum()).log().alias('strategy'),
        pl.col('fee').sum().alias('fee'),])
    .sort(['ts'])
    .join(
        df
            .filter(pl.col('symbol') == 'btc')
            .select([pl.col('ts'),pl.col('ret').alias('ref')]),
        on='ts',
        how='inner')
    .with_columns([
        (pl.col('strategy') - pl.col('ref')).alias('perf')])
    .with_columns([
        pl.when(pl.col('perf').shift(1).rolling_mean(20) >= 0)
            .then(pl.col('perf'))
            .otherwise(0)
            .alias('cond')])
    .with_columns([
        pl.col('cond').cum_sum().alias('equity')])
    .to_pandas()
    .set_index('ts')
    .sort_index())

res.write_parquet(output_eval_file)

FileNotFoundError: No such file or directory (os error 2): chronos-2-predictions.parquet

This error occurred with the following context stack:
	[1] 'parquet scan'
	[2] 'sink'


In [83]:
res = pl.read_parquet(output_eval_file)

fig, [ax1, ax2, ax3] = plt.subplots(3,1,figsize=(18,10), sharex=True)

((np.exp(res.strategy) - 1) * 100).rolling(30).mean().plot(y='strategy',ax=ax1,color='gray')
((np.exp(res.perf) - 1) * 100).rolling(30).mean().plot(y='perf',ax=ax1,color='black')
ax1.axhline(0)
res.equity.plot(y='equity',ax=ax2)
(res.fee * 100).rolling(10).mean().plot(y='fee',ax=ax3)

FileNotFoundError: No such file or directory (os error 2): chronos-2-eval.parquet

This error occurred with the following context stack:
	[1] 'parquet scan'
	[2] 'sink'
